In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 4b: Pohon yang digabung

Cerita lengkapnya ada di `README.md`. Jawab dulu, baru jalankan kode.

Paket adalah bundel kode siap pakai. Pustaka adalah kumpulan fungsi di dalam paket. pandas dan scikit-learn adalah pustaka yang dipakai notebook ini.

Klasifikasi, dari bagian sebelumnya, artinya menebak golongan, bukan angka. Pohon keputusan artinya bagan pertanyaan ya atau tidak, satu kolom tiap pertanyaan. Dua kalimat itu mengingat arti yang sudah ada, bukan istilah baru di notebook ini.

Tiga pemilah berkata: matang, mentah, matang. Golongan mana yang menang?


In [2]:
# Satu ide: hitung suara terbanyak pada tiga pilihan tangan.
# Counter menghitung berapa kali tiap kata muncul. Itu alat bawaan Python, bukan istilah baru.
from collections import Counter

suara = ["matang", "mentah", "matang"]
hitung = Counter(suara)
terbanyak = hitung.most_common(1)[0][0]
print("suara", suara)
print("hitung", dict(hitung))
print("terbanyak", terbanyak)


suara ['matang', 'mentah', 'matang']
hitung {'matang': 2, 'mentah': 1}
terbanyak matang


## Kasus nyata

Tabel anggur ini sudah ikut terpasang di scikit-learn (`load_wine`). Sel di bawah tidak mengunduh apa pun. Ini bukan data asli perusahaan.

Isinya hasil analisis kimia anggur dari daerah yang sama di Italia, dari tiga cultivars. Nama golongannya `class_0`, `class_1`, dan `class_2`: tiga kelompok, bukan peringkat dari buruk ke baik.


In [3]:
# Satu ide: panggil pandas, pustaka untuk tabel.
import pandas as pd


In [4]:
# Satu ide: muat anggur dari berkas yang sudah terpasang.
from sklearn.datasets import load_wine

anggur = load_wine()


In [5]:
# Satu ide: simpan ukuran kimia dan nama golongan dalam satu tabel.
# feature_names adalah nama kolom bawaan dataset, bukan istilah baru.
tabel = pd.DataFrame(anggur.data, columns=anggur.feature_names)
tabel["golongan"] = anggur.target_names[anggur.target]
print("baris", int(tabel.shape[0]))
print("kolom_ukuran", int(anggur.data.shape[1]))
print("class_0", int((tabel["golongan"] == "class_0").sum()))
print("class_1", int((tabel["golongan"] == "class_1").sum()))
print("class_2", int((tabel["golongan"] == "class_2").sum()))


baris 178
kolom_ukuran 13
class_0 59
class_1 71
class_2 48


In [6]:
# Satu ide: sisihkan data uji. test_size=0.25 berarti seperempat baris menjadi data uji.
# stratify menjaga proporsi tiap golongan di kedua bagian mirip dengan tabel penuh.
# random_state=SEED memakai angka tetap dari sel pertama.
from sklearn.model_selection import train_test_split

ukuran = tabel[list(anggur.feature_names)]
golongan = tabel["golongan"]
ukuran_latih, ukuran_uji, golongan_latih, golongan_uji = train_test_split(
    ukuran,
    golongan,
    test_size=0.25,
    random_state=SEED,
    stratify=golongan,
)
print("baris_latih", int(len(golongan_latih)))
print("baris_uji", int(len(golongan_uji)))


baris_latih 133
baris_uji 45


In [7]:
# Satu ide: baseline selalu menebak golongan yang paling banyak di data latih.
# DummyClassifier adalah nama baseline ini di scikit-learn.
# fit artinya model belajar dari data latih.
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline.fit(ukuran_latih, golongan_latih)


In [8]:
# Satu ide: hitung tebakan baseline yang salah.
# predict artinya model menebak golongan baris yang diberikan.
tebakan = baseline.predict(ukuran_uji)
salah_uji_baseline = int((tebakan != golongan_uji.to_numpy()).sum())
print("baseline_selalu", tebakan[0])
print("salah_uji_baseline", salah_uji_baseline)
assert baseline.strategy == "most_frequent", (
    "Baseline harus menebak golongan yang paling banyak."
)
assert salah_uji_baseline == 27, (
    "Salah uji baseline harus 27. Angka ini berubah bila strategi baseline diganti."
)


baseline_selalu class_1
salah_uji_baseline 27


In [9]:
# Satu ide: latih satu pohon keputusan tanpa batas kedalaman.
from sklearn.tree import DecisionTreeClassifier

pohon = DecisionTreeClassifier(random_state=SEED)
pohon.fit(ukuran_latih, golongan_latih)


In [10]:
# Satu ide: hitung tebakan salah satu pohon itu.
salah_latih_pohon = int(
    (pohon.predict(ukuran_latih) != golongan_latih.to_numpy()).sum()
)
salah_uji_pohon = int(
    (pohon.predict(ukuran_uji) != golongan_uji.to_numpy()).sum()
)
print("salah_latih_pohon", salah_latih_pohon)
print("salah_uji_pohon", salah_uji_pohon)
print("pohon_mengalahkan_baseline", salah_uji_pohon < salah_uji_baseline)


salah_latih_pohon 0
salah_uji_pohon 2
pohon_mengalahkan_baseline True


In [11]:
# Satu ide: latih hutan acak pada data latih yang sama.
from sklearn.ensemble import RandomForestClassifier

hutan = RandomForestClassifier(random_state=SEED)
hutan.fit(ukuran_latih, golongan_latih)


In [12]:
# Satu ide: hitung tebakan salah hutan acak.
salah_latih_hutan = int(
    (hutan.predict(ukuran_latih) != golongan_latih.to_numpy()).sum()
)
salah_uji_hutan = int(
    (hutan.predict(ukuran_uji) != golongan_uji.to_numpy()).sum()
)
# n_estimators adalah banyaknya pohon.
print("banyak_pohon", int(hutan.n_estimators))
print("salah_latih_hutan", salah_latih_hutan)
print("salah_uji_hutan", salah_uji_hutan)
assert int(hutan.n_estimators) == 100, (
    "Hutan harus memakai seratus pohon, bukan satu."
)
assert salah_uji_hutan == 0, "Salah uji hutan pada pecahan ini harus 0."


banyak_pohon 100
salah_latih_hutan 0
salah_uji_hutan 0


In [13]:
# Satu ide: hitung suara tiap pohon pada baris pertama data uji.
# iloc mengambil baris menurut urutan di tabel ini. Angka 0 berarti baris pertama.
# estimators_ adalah daftar pohon di dalam hutan. to_numpy mengubah baris menjadi angka,
# karena tiap pohon di dalam hutan menerima angka, bukan nama kolom.
# classes_ adalah daftar nama golongan menurut scikit-learn.
from collections import Counter

baris = ukuran_uji.iloc[[0]].to_numpy()
suara_pohon = []
for satu in hutan.estimators_:
    kode = int(satu.predict(baris)[0])
    suara_pohon.append(hutan.classes_[kode])
hitung_suara = Counter(suara_pohon)
print("jumlah_pohon", len(suara_pohon))
for nama in hutan.classes_:
    print("suara", nama, int(hitung_suara[nama]))
print("golongan_benar", golongan_uji.iloc[0])


jumlah_pohon 100
suara class_0 95
suara class_1 2
suara class_2 3
golongan_benar class_0


In [14]:
# Satu ide: bandingkan peluang rata-rata hutan dengan bagian suara pada baris yang sama.
# predict_proba memberi peluang tiap golongan, antara 0 dan 1.
peluang_baris = hutan.predict_proba(ukuran_uji.iloc[[0]])[0]
jumlah = len(hutan.estimators_)
sama = True
for nama, nilai in zip(hutan.classes_, peluang_baris):
    bagian = hitung_suara[nama] / jumlah
    print("peluang", nama, float(nilai))
    print("bagian_suara", nama, bagian)
    if float(nilai) != bagian:
        sama = False
print("tebakan", hutan.predict(ukuran_uji.iloc[[0]])[0])
print("sama_pada_baris_ini", sama)


peluang class_0 0.95
bagian_suara class_0 0.95
peluang class_1 0.02
bagian_suara class_1 0.02
peluang class_2 0.03
bagian_suara class_2 0.03
tebakan class_0
sama_pada_baris_ini True


In [15]:
# Satu ide: latih gradient boosting pada data latih yang sama.
from sklearn.ensemble import GradientBoostingClassifier

boosting = GradientBoostingClassifier(random_state=SEED)
boosting.fit(ukuran_latih, golongan_latih)


In [16]:
# Satu ide: cetak tombol langkah dan tebakan salah gradient boosting.
# learning_rate mengalikan sumbangan tiap pohon baru supaya langkahnya kecil.
print("learning_rate", float(boosting.learning_rate))
print("banyak_tahap", int(boosting.n_estimators))
salah_latih_boosting = int(
    (boosting.predict(ukuran_latih) != golongan_latih.to_numpy()).sum()
)
salah_uji_boosting = int(
    (boosting.predict(ukuran_uji) != golongan_uji.to_numpy()).sum()
)
print("salah_latih_boosting", salah_latih_boosting)
print("salah_uji_boosting", salah_uji_boosting)


learning_rate 0.1
banyak_tahap 100
salah_latih_boosting 0
salah_uji_boosting 1


In [17]:
# Satu ide: bandingkan tahap pertama dan tahap terakhir pada data uji.
# staged_predict memberi tebakan setelah tiap pohon ditambahkan.
tahap = list(boosting.staged_predict(ukuran_uji))
salah_uji_tahap_pertama = int((tahap[0] != golongan_uji.to_numpy()).sum())
salah_uji_tahap_terakhir = int((tahap[-1] != golongan_uji.to_numpy()).sum())
print("salah_uji_tahap_pertama", salah_uji_tahap_pertama)
print("salah_uji_tahap_terakhir", salah_uji_tahap_terakhir)
print(
    "terakhir_lebih_sedikit",
    salah_uji_tahap_terakhir < salah_uji_tahap_pertama,
)


salah_uji_tahap_pertama 14
salah_uji_tahap_terakhir 1
terakhir_lebih_sedikit True


In [18]:
# Satu ide: pada data uji ini, model mana yang salahnya paling sedikit.
pasangan = {
    "pohon": salah_uji_pohon,
    "hutan_acak": salah_uji_hutan,
    "gradient_boosting": salah_uji_boosting,
}
paling_sedikit = min(pasangan, key=pasangan.get)
print("paling_sedikit_salah_uji", paling_sedikit)


paling_sedikit_salah_uji hutan_acak
